### Transformers is a better alternative then LSTM and powers modern LLMs

##### LSTM goes word by word , BERT considers whole sentence

In [1]:
import pandas as pd
from transformers import BertTokenizer , BertForSequenceClassification , Trainer , TrainingArguments
from sklearn.model_selection import train_test_split
from datasets import Dataset 

d:\LSTM\.venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import warnings
warnings.filterwarnings('ignore')

#### Data Preprocess

In [3]:
df = pd.read_csv('D:\LSTM\Dataset\IMDB Dataset.csv')

In [8]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [4]:
df['sentiment'] = df['sentiment'].map({'positive':1,'negative':0})

In [5]:
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,1
1,A wonderful little production. <br /><br />The...,1
2,I thought this was a wonderful way to spend ti...,1
3,Basically there's a family where a little boy ...,0
4,"Petter Mattei's ""Love in the Time of Money"" is...",1


In [17]:
df = df.sample(500, random_state=42)

In [18]:
df.shape

(500, 2)

In [19]:
train_text , test_text , train_label , test_label = train_test_split(
    df['review'].tolist(),
    df['sentiment'].to_list(),
    test_size = 0.2,
    random_state = 42
)

#### Loading BERT

In [9]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [20]:
train_encodings = tokenizer(
    train_text,
    truncation = True,
    padding = True,
    max_length = 128
)

test_encodings = tokenizer(
    test_text,
    truncation = True,
    padding = True,
    max_length = 128
)

In [21]:
train_dataset = Dataset.from_dict({

    "input_ids": train_encodings["input_ids"],

    "attention_mask": train_encodings["attention_mask"],

    "labels": train_label
})

test_dataset = Dataset.from_dict({

    "input_ids": test_encodings["input_ids"],

    "attention_mask": test_encodings["attention_mask"],

    "labels": test_label
})

In [12]:

model = BertForSequenceClassification.from_pretrained(

    "bert-base-uncased",

    num_labels=2
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5432.82it/s]
[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoi

In [13]:
training_args = TrainingArguments(

    output_dir="D:\LSTM\BERT",

    num_train_epochs=2,

    per_device_train_batch_size=8,

    per_device_eval_batch_size=8,

    eval_strategy="epoch",

    save_strategy="epoch",

)

In [22]:
trainer = Trainer(

    model=model,

    args=training_args,

    train_dataset=train_dataset,

    eval_dataset=test_dataset
)

In [23]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,0.370947
2,No log,0.524229


Writing model shards: 100%|██████████| 1/1 [00:04<00:00,  4.04s/it]


TrainOutput(global_step=100, training_loss=0.36140205383300783, metrics={'train_runtime': 882.5101, 'train_samples_per_second': 0.907, 'train_steps_per_second': 0.113, 'total_flos': 52622211072000.0, 'train_loss': 0.36140205383300783, 'epoch': 2.0})

In [28]:
import tensorflow as tf
import torch

In [36]:


text = "this movie was so bad i will never watch it again......"

inputs = tokenizer(

    text,

    return_tensors="pt",

    truncation=True,

    padding=True,

    max_length=64
)

with torch.no_grad():

    outputs = model(**inputs)

    logits = outputs.logits

    probabilities = torch.nn.functional.softmax(

        logits,

        dim=1
    )

predicted_class = torch.argmax(

    probabilities,

    dim=1

).item()

confidence = probabilities[0][predicted_class].item() * 100

if predicted_class == 1:

    print("Positive Review")

else:

    print("Negative Review")

print(f"Confidence: {confidence:.2f}%")

Negative Review
Confidence: 97.92%
